# USDC-WETH Pool Unsupervised Regime Classification

## Features

- eth_ret_1h
- eth_ret_24h
- eth_rv_4h
- eth_rv_24h
- pool_tvl
- pool_volume_1h
- vol_tvl_ratio_1h
- fee_apr_24h
- swap_count_1h
- avg_trade_size_1h


In [78]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [79]:
swaps = pd.read_parquet("../data/processed/usdcweth_swaps.parquet")
swaps['block_timestamp'] = pd.to_datetime(swaps['block_timestamp'], utc=True)
swaps = swaps.sort_values('block_timestamp')
swaps.set_index("block_timestamp",inplace=True)

In [80]:
swaps.head()

,block_number,tx_hash,log_index,tx_index,sender,recipient,amount0,amount1,sqrtPriceX96,liquidity,tick
block_timestamp,,,,,,,,,,,
2021-05-05 01:56:23+00:00,12371376,ce7c3c307d820785caa12938012372fc9366a614a6aacf...,26,20,0xE592427A0AEce92De3Edee1F18E0157C05861564,0x5Eefc9306f11a824762CcDaedaC41049EFc7fcc8,-329608,100000000000000,1377932816571815120446551350158799,4303369674465501,195285.0
2021-05-05 08:23:26+00:00,12373132,9a1c51b0bffbf840948f3b6e3f3e495ba1cd3fa64854f9...,192,172,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xE592427A0AEce92De3Edee1F18E0157C05861564,-164694492,50000000000000000,1378850591292581266780357299649652,4303369674465501,195298.0
2021-05-05 09:50:51+00:00,12373520,c58715c62a5bf70a6ca09f0e51546d6cad76c8d4fff036...,8,18,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xFB36693ac2DBE8DFC9f89dD0de6015c6Ea66b0BF,-329169,100000000000000,1378852426842022799073024911548633,4303369674465501,195298.0
2021-05-05 11:59:57+00:00,12374077,288c21b8b4fbf449b1d086a06e43b124ac2bc088c3f536...,86,102,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xFB36693ac2DBE8DFC9f89dD0de6015c6Ea66b0BF,2,-329169,1378852426842016741051966412054516,4304946248093346,195298.0
2021-05-05 12:56:56+00:00,12374320,67502d8ba373287f6d301f6baa77c5a5f4c80d0753c34c...,257,115,0xE592427A0AEce92De3Edee1F18E0157C05861564,0x22F9dCF4647084d6C31b2765F6910cd85C178C18,1559137299,-467880854065813753,1370241555019945317645788135487819,4304946248093346,195173.0


In [81]:
df = swaps.copy()

In [83]:
df = df[['sqrtPriceX96', 'amount0', 'amount1']].dropna().copy()

### Price Features

In [84]:
USDC_DEC = 6
WETH_DEC = 18
Q192 = 2 ** 192

sqrtP = df["sqrtPriceX96"].astype(float)
df["price_weth_per_usdc"] = (sqrtP**2 / Q192)*(10 ** (USDC_DEC - WETH_DEC))
df["price_usdc_per_weth"] = 1/df["price_weth_per_usdc"]

### Volume Features

In [96]:
df["sqrtPriceX96"] = df["sqrtPriceX96"].astype(float)
df["amount0"] = df["amount0"].astype(float)
df["amount1"] = df["amount1"].astype(float)
df["abs_amount0"] = df["amount0"].abs()
df["abs_amount1"] = df["amount1"].abs()

In [97]:
df["volume_usd"] = df["abs_amount0"] + df["abs_amount1"] * df["price_usdc_per_weth"]

In [98]:
df.head()

,sqrtPriceX96,amount0,amount1,price_weth_per_usdc,price_usdc_per_weth,abs_amount0,abs_amount1,volume_usd
block_timestamp,,,,,,,,
2021-05-05 01:56:23+00:00,1.377933e+33,-3.296080e+05,1.000000e+14,0.000302,3306.001763,3.296080e+05,1.000000e+14,330600176292241536.0
2021-05-05 08:23:26+00:00,1.378851e+33,-1.646945e+08,5.000000e+16,0.000303,3301.602222,1.646945e+08,5.000000e+16,165080111081095462912.0
2021-05-05 09:50:51+00:00,1.378852e+33,-3.291690e+05,1.000000e+14,0.000303,3301.593431,3.291690e+05,1.000000e+14,330159343134100032.0
2021-05-05 11:59:57+00:00,1.378852e+33,2.000000e+00,-3.291690e+05,0.000303,3301.593431,2.000000e+00,3.291690e+05,1086782210.200012
2021-05-05 12:56:56+00:00,1.370242e+33,1.559137e+09,-4.678809e+17,0.000299,3343.219561,1.559137e+09,4.678809e+17,1564228423644219703296.0


### Resample 1 hour buckets

In [99]:
hourly_swaps = df.resample("1h").agg(
    price_close=("price_usdc_per_weth", "last"),
    price_open=("price_usdc_per_weth", "first"),
    price_high=("price_usdc_per_weth", "max"),
    price_low=("price_usdc_per_weth", "min"),
    swap_count=("volume_usd", "count"),
    volume_usd_1h=("volume_usd", "sum"),
    avg_trade_size_usd=("volume_usd", "mean"),
)

### Returns and realized vol

In [102]:
# make sure the column is numeric
hourly_swaps["price_close"] = pd.to_numeric(
    hourly_swaps["price_close"], errors="coerce"
)

hourly_swaps["log_price"] = np.log(hourly_swaps["price_close"])
hourly_swaps["ret_1h"] = hourly_swaps["log_price"].diff()


In [103]:
hourly_swaps["rv_4h"] = (
    hourly_swaps["ret_1h"]
    .rolling(4)
    .apply(lambda x: np.sqrt((x**2).sum()), raw=True)
)

In [104]:
hourly_swaps["rv_24h"] = (
    hourly_swaps["ret_1h"]
    .rolling(24)
    .apply(lambda x: np.sqrt((x**2).sum()), raw=True)
)

In [105]:
hourly_swaps["ma_4h"] = hourly_swaps["price_close"].rolling(4).mean()
hourly_swaps["ma_24h"] = hourly_swaps["price_close"].rolling(24).mean()
hourly_swaps["ma_ratio"] = hourly_swaps["ma_4h"] / hourly_swaps["ma_24h"]

### Now lets approximate net liquidity using the mints and burns 

In [114]:
mints = pd.read_parquet("../data/processed/usdcweth_mints.parquet")
burns = pd.read_parquet("../data/processed/usdcweth_burns.parquet")

In [115]:
mints.head()

,block_timestamp,block_number,tx_hash,log_index,tx_index,sender,owner,tickLower,tickUpper,amount,amount0,amount1
0,2022-11-16 21:35:47,15985300,f2117efba6da137734545b19bfc58c25cdd5d75ef4a8df...,104,42,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,196200.0,214140.0,437629118975,5416821,4616252638023322
1,2022-11-16 22:27:35,15985558,e3eee6f0676d89a0f2398dfd698c42c584afec4c4b18e4...,95,62,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,204000.0,206760.0,46867685282969960,108451499348,90027183494939958762
2,2022-11-16 16:19:23,15983731,cff26ea10d7aad189ab3800e14df4041d2bcd4a986265c...,4,0,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,205380.0,205440.0,75617578496708709004,1889892676794,4968200383587929499950
3,2022-11-16 16:23:11,15983750,685524cbf119f89cea053bbadfd94051355f57332beaf6...,472,262,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,-887220.0,887220.0,346704704341,12006422,10011654763829435
4,2022-11-16 16:34:47,15983808,398c9be046cb2b08bbfab6b67c66c39e26d753b6dc0ee3...,4,0,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,205380.0,205440.0,40958090675352010780,1532397444687,2266967543385215461963


In [116]:
burns.head()

,block_timestamp,block_number,tx_hash,log_index,tx_index,owner,tickLower,tickUpper,amount,amount0,amount1
0,2022-11-16 21:41:59,15985331,7fb8fe34835a9bed2abd9194fe16ba2d4035e32703e55b...,201,137,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,194820.0,208620.0,0,0,0
1,2022-11-16 21:42:23,15985333,3b71fdfd9b24e64f5665c3393b40eeeffddddfaa78ed6e...,171,98,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,197400.0,211260.0,0,0,0
2,2022-11-16 21:42:23,15985333,f1f9b2ca96b498d10e185e1e9e9623d81ffd7b2ae914e0...,279,141,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,197400.0,211260.0,0,0,0
3,2022-11-16 21:42:35,15985334,051e29851f59ac63920a58305ae18144dd8ef2c526c511...,195,122,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,197220.0,211140.0,0,0,0
4,2022-11-16 21:45:11,15985347,406f257b0701faf30460201aa748a31ad7785d287f0569...,153,114,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,198900.0,203160.0,57731509032457,0,285551285939997447


In [117]:
# ensure both are in proper datetime and datetime index
for df in (mints, burns):
    df["ts"] = pd.to_datetime(df["block_timestamp"], unit="s")
    df.set_index("block_timestamp", inplace=True)